In [10]:
import os 
print([x for x in os.listdir('scripts')])

['deep.sh', '.ipynb_checkpoints', 'run_paml_null.sh', 'M0_null.ctl', 'update_taxonomy.sh', 'split_diamond.sh', 'mcmctree-outBV.ctl', 'gffread.sh', 'run_busco_phylogenomics.sh', 'diamond_clustered_all_dipt.sh', 'get_taxid_accessions.sh', 'deeplocsh.sh', 'update_taxonomy.py', 'query_clustered_nr_protein.sh', 'run_deeploc.py', 'interproscan.sh', 'download_genomes.sh', 'diptera_diamond_processing.py', 'mcmctree.sh', 'trimAl.sh', 'run_busco.sh', 'trimal_python.py', 'run_ONT_mapper.sh', 'download_clustered_nr.sh', 'iqtree.sh', 'M0.ctl', 'insect.dat', 'mcmctree.nw', 'iqtree_no_partition.sh', 'diamond_clustered.sh', 'run_phylopipe.sh', 'deseq2Counts_copro.R', 'diamond_sciarid.sh', 'mcmctree.ctl', 'iqtree_partition.sh', 'run_paml.sh']


In [1]:
import pandas as pd
from Bio import SeqIO

In [2]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_family_taxid(taxid):
    """
    Return the family taxid for a given NCBI taxid.

    Parameters
    ----------
    taxid : int

    Returns
    -------
    family
    """
    lineage = ncbi.get_lineage(taxid)
    ranks = ncbi.get_rank(lineage)
  

    family_taxid = next(
        (tid for tid in lineage if ranks.get(tid) == "family"),
        None
    )
    
    if family_taxid is None:
        return None, None

    family_name = ncbi.get_taxid_translator([family_taxid])[family_taxid]
    return family_name

/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
df=pd.read_csv("datasets/annotated_dipteran_genomes_6_7_2026.tsv",sep="\t")

In [4]:
df=df[df['Organism Name']!='Rhagoletis zephyria']
df=df[df['Organism Name']!='Bradysia coprophila']

In [5]:
for index, row in df.iterrows():
    df.loc[index,'genus']=row['Organism Name'].split(" ")[0]
## select only some members of genera Anopheles and Drosophila for analysis 
add=['Anopheles cruzii','Anopheles darlingi', 'Anopheles ziemanni','Anopheles gambiae','Anopheles funestus', 'Drosophila grimshawi',
    'Drosophila innubila','Drosophila virilis','Drosophila pseudoobscura','Drosophila arizonae','Drosophila willistoni','Drosophila ananassae','Drosophila kikkawai','Drosophila suzukii','Drosophila melanogaster','Drosophila grimshawi']


td=add+list(df[~df.genus.isin(["Anopheles","Drosophila"])]["Organism Name"])
td=df[df["Organism Name"].isin(td)]["Assembly Accession"]
len(td)

53

In [6]:
for index, row in df.iterrows():
    df.loc[index,'family']=get_family_taxid(row['Organism Taxonomic ID'])

In [18]:
import os
for x in set(df['Assembly Accession']):
    if 'genomic.gff' not in os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{x}/") or "protein.faa" not in os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{x}"):
        !sh scripts/download_genomes.sh "$x"

In [47]:
from Bio import SeqIO
from collections import defaultdict
def longest_iso(ac):
    gff=pd.read_csv(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{ac}/genomic.gff",sep="\t",comment="#",header=None)
    def get_gene(x):
        return gff[gff[8].str.contains(x)].iloc[0,8].split(";gene=")[1].split(";")[0]
        
    input_file = open(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{ac}/protein.faa") 
    fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
    
    cds=gff[gff[2]=='CDS'].copy()
    cds['prot']=[x.split(";")[0].split("-")[1] for x in cds[8]]
    cds['gene']=[x.split(";gene=")[1].split(";")[0] for x in cds[8]]

    cds_gene={}
    gene_cds=defaultdict(set)
    for index, row in cds.iterrows():
        cds_gene[row.prot]=row.gene
        gene_cds[row.gene].add(row.prot)

    cds_len={x:len(str(fasta[x].seq)) for x in fasta}
    longest = []

    for gene, proteins in gene_cds.items():
        max_len=max([cds_len[x] for x in proteins])
        longest.append([x for x in proteins if cds_len[x]==max_len][0])
    longest_records = []

    for x in longest:
        record = fasta[x]
        record.id = f"{ac};{x}"
        record.name = f"{ac};{x}"
        record.description = f"{ac};{x}"
        longest_records.append(record)

    
    SeqIO.write(
        longest_records,
        f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{ac}/longest_isoform.fasta",
        "fasta"
    )

In [47]:
!mkdir "$SCRATCH/extavour_lab/Everyone/Rishabh/diamond_clustered_dipteran_results"

In [1]:
!mkdir /n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_dipteran_results_split

In [1]:
import os
for x in set(df['Assembly Accession']):
    longest_iso(x)
    !sbatch diamond_clustered_all_dipt.sh "$x"

In [7]:
import subprocess
from pathlib import Path


def run_split_diamond(genome_accession):
    """
    Run scripts/split_diamond.sh for a given genome accession.

    Example
    -------
    run_split_diamond("GCF_030788295.1")
    """

    split_dir = Path(
        "/n/netscratch/extavour_lab/Everyone/Rishabh/"
        "diamond_clustred_dipteran_results_split"
    ) / genome_accession

    diamond_out = Path(
        "/n/netscratch/extavour_lab/Everyone/Rishabh/"
        "diamond_clustered_dipteran_results"
    ) / f"{genome_accession}_out"

    cmd = [
        "sh",
        "scripts/split_diamond.sh",
        str(split_dir),
        str(diamond_out),
    ]

    print("Running:")
    print(" ".join(cmd))

    subprocess.run(cmd, check=True)

In [ ]:
for index, row in dfresult.iterrows():
    non_dipt=(row.n_out_fam_diptera==0)&(row.evalue<1e-5)
    alien=row.AI>10
    if alien:
        dfresult.loc[index,'type']='alien'
    elif non_dipt:
        dfresult.loc[index,'type']='non_dipt'
    elif 'Diptera' in str(row.slineage):
        dfresult.loc[index,'type']='dipt'
    elif str(row.sseqid)=='nan':
        dfresult.loc[index,'type']='orphan'
    dfresult.loc[index,'HGT']=alien|non_dipt
    

In [27]:
!mkdir outputs/diptera_wide_screen

In [29]:
for index, row in df[df['Assembly Accession'].isin(td)].iterrows():
    genome=row['Assembly Accession']
    print(genome)
    family=row['family']
    # run_split_diamond(genome)
    subprocess.run(['python3','diptera_diamond_processing.py', genome, family])

GCF_024334085.1


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


GCF_000001215.4


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


GCF_002204515.2


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


... (96 outputs left out)

GCF_047663935.1


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


GCF_009176525.2


/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [14]:
from collections import Counter
diptera_df=df[df['Assembly Accession'].isin(td)]
diptera_df=diptera_df.loc[:,['Assembly Accession','Organism Name','Organism Taxonomic ID','family']]
diptera_df.columns=['Genome','Organism Name','taxid','family']
for index, row in diptera_df.iterrows():
    g=row.Genome
    input_file=open(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{g}/longest_isoform.fasta")
    fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
    diptera_df.loc[index,'total_proteins']=len(fasta)
    result_df=pd.read_csv(f"outputs/diptera_wide_screen/{g}.tsv",sep='\t',index_col=0)
    c=Counter(result_df.type)
    for ci in c:
        diptera_df.loc[index, ci]=c[ci]
        


In [15]:
diptera_df=diptera_df.set_index("Genome")

In [16]:
index='GCF_014529535.2'
input_file=open(f"ncbi_dataset/data/GCF_014529535.2/longest_isoform.fasta")
fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))

diptera_df.loc[index,'total_proteins']=len(fasta)
result_df=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep='\t',index_col=0)
c=Counter(result_df.type)
for ci in c:
    diptera_df.loc[index, ci]=c[ci]
diptera_df.loc[index,['Organism Name','taxid','family']]='Bradysia coprophila',38358,'Sciaridae'

In [17]:
diptera_df=diptera_df.fillna(0)
diptera_df['HGT']=diptera_df['alien']+diptera_df['non_dipt']
diptera_df['p_HGT']=diptera_df['HGT']/diptera_df['total_proteins']
diptera_df=diptera_df.sort_values('family')
diptera_df['none']=diptera_df['total_proteins']-diptera_df['dipt']-diptera_df['orphan']-diptera_df['alien']-diptera_df['non_dipt']

In [18]:
!mkdir -p "outputs/SI_tables/"
diptera_df.to_csv("outputs/SI_tables/SI_table_1.tsv",sep="\t")

In [19]:
df_family=pd.DataFrame()
for index, df in diptera_df.groupby('family'):
    n=df.shape[0]
    df_family.loc[index+f" ({n})",['n_HGT','p_HGT']]=round(df.HGT.mean(),0),round(100*df.p_HGT.mean(),1)

In [21]:
## diptera tree from https://doi.org/10.1073/pnas.1012675108
a = "(((Psychodidae,(Ceratopogonidae,Culicidae)),((Sciaridae,Cecidomyiidae),(Stratiomyidae,(Syrphidae,((Tephritidae,Diopsidae),(Drosophilidae,(Glossinidae,(Muscidae,Calliphoridae)))))))));"
from ete3 import Tree
a=Tree(a)
print(a)



         /-Psychodidae
      /-|
     |  |   /-Ceratopogonidae
     |   \-|
     |      \-Culicidae
-- /-|
     |      /-Sciaridae
     |   /-|
     |  |   \-Cecidomyiidae
      \-|
        |   /-Stratiomyidae
        |  |
         \-|   /-Syrphidae
           |  |
            \-|      /-Tephritidae
              |   /-|
              |  |   \-Diopsidae
               \-|
                 |   /-Drosophilidae
                  \-|
                    |   /-Glossinidae
                     \-|
                       |   /-Muscidae
                        \-|
                           \-Calliphoridae


In [22]:
df_family.index=[x.replace("(","_").replace(')','_') for x in df_family.index]
df_family

,n_HGT,p_HGT
Calliphoridae _2_,36.0,0.3
Cecidomyiidae _2_,491.0,3.3
Ceratopogonidae _1_,62.0,0.6
Culicidae _16_,473.0,2.9
Diopsidae _1_,219.0,0.9
Drosophilidae _9_,77.0,0.5
Glossinidae _1_,23.0,0.2
Muscidae _4_,56.0,0.3
Psychodidae _3_,126.0,1.1
Sciaridae _1_,1721.0,10.5


In [23]:
df_family.shape[0]

13

In [25]:
%%bash
tar -czf diptera_wide_screen.tar.gz \
  -C outputs/ \
  diptera_wide_screen